# Filter spreadsheet by basenames

This notebook:
1. reads a comma-separated list of basenames from `converted_basenames_comma_list.txt`;
2. reads the source Excel file;
3. filters all rows where column `basename` is in the list;
4. writes a new Excel file with:
   - `Filtered_Records`
   - `Missing_basenames`
   - `Summary`

Important fix: the comma list is read as raw text and basenames are extracted as full numeric tokens, not character-by-character.


In [1]:
from pathlib import Path
from decimal import Decimal, InvalidOperation, ROUND_HALF_UP
import re
import pandas as pd


In [2]:
BASE_DIR = Path(".")
SOURCE_XLSX = BASE_DIR / "visi_zive_irasai_annot-Darb_26_04_16_atrankai.xlsx"
COMMA_LIST_TXT = BASE_DIR / "converted_basenames_comma_list.txt"
OUTPUT_XLSX = BASE_DIR / "filtered_records_by_basenames.xlsx"

BASENAME_COLUMN = "basename"

print("Source Excel:", SOURCE_XLSX.resolve())
print("Comma list:", COMMA_LIST_TXT.resolve())
print("Output Excel:", OUTPUT_XLSX.resolve())


Source Excel: /home/kesju/DI/2025_ZIVEO/PROJECT_TRAIN_UNET/1_PREPARE_TRAIN_UNET_DATA/CONVERT_ZIVE_TO&FROM_NPY/visi_zive_irasai_annot-Darb_26_04_16_atrankai.xlsx
Comma list: /home/kesju/DI/2025_ZIVEO/PROJECT_TRAIN_UNET/1_PREPARE_TRAIN_UNET_DATA/CONVERT_ZIVE_TO&FROM_NPY/converted_basenames_comma_list.txt
Output Excel: /home/kesju/DI/2025_ZIVEO/PROJECT_TRAIN_UNET/1_PREPARE_TRAIN_UNET_DATA/CONVERT_ZIVE_TO&FROM_NPY/filtered_records_by_basenames.xlsx


In [3]:
def normalize_basename(value) -> str:
    if pd.isna(value):
        return ""

    s = str(value).strip()
    s = s.replace("\ufeff", "").strip()

    if s.lower().endswith(".npy"):
        s = s[:-4].strip()

    m = re.search(r"\d+(?:\.\d+)?", s)
    if not m:
        return s

    num = m.group(0)

    try:
        d = Decimal(num)
        return f"{d.quantize(Decimal('0.001'), rounding=ROUND_HALF_UP):f}"
    except InvalidOperation:
        return num


def read_basename_comma_list(path: Path) -> list[str]:
    text = path.read_text(encoding="utf-8-sig")

    raw_tokens = re.findall(r"\d+(?:\.\d+)?", text)
    basenames = [normalize_basename(x) for x in raw_tokens]

    seen = set()
    unique = []
    for b in basenames:
        if b and b not in seen:
            unique.append(b)
            seen.add(b)

    return unique


In [4]:
requested_basenames = read_basename_comma_list(COMMA_LIST_TXT)

print("Basenames in comma list:", len(requested_basenames))
print(",".join(requested_basenames[:10]) + ("..." if len(requested_basenames) > 10 else ""))

assert requested_basenames, "No basenames were found in the comma-list file."
assert all("." in b for b in requested_basenames), "Some parsed items do not look like basenames."
assert not any(len(b) == 1 for b in requested_basenames), "Parsing error: single-character tokens found."


Basenames in comma list: 35
1670188.752,1670183.201,1630715.197,1630733.908,1630736.382,1630797.676,1630959.214,1630771.589,1630758.549,1631057.107...


In [5]:
df = pd.read_excel(SOURCE_XLSX, dtype={BASENAME_COLUMN: str})

if BASENAME_COLUMN not in df.columns:
    raise ValueError(f"Column '{BASENAME_COLUMN}' was not found. Available columns: {list(df.columns)}")

df["_basename_norm"] = df[BASENAME_COLUMN].apply(normalize_basename)

requested_set = set(requested_basenames)
filtered = df[df["_basename_norm"].isin(requested_set)].copy()

found_set = set(filtered["_basename_norm"])
missing = [b for b in requested_basenames if b not in found_set]

filtered_out = filtered.drop(columns=["_basename_norm"])

missing_df = pd.DataFrame({"basename": missing})
summary_df = pd.DataFrame({
    "Parameter": [
        "Source Excel",
        "Comma-list file",
        "Basename column",
        "Basenames requested",
        "Rows copied",
        "Missing basenames",
        "Output Excel",
    ],
    "Value": [
        str(SOURCE_XLSX),
        str(COMMA_LIST_TXT),
        BASENAME_COLUMN,
        len(requested_basenames),
        len(filtered_out),
        len(missing),
        str(OUTPUT_XLSX),
    ],
})

print("Rows copied:", len(filtered_out))
print("Missing basenames:", len(missing))
if missing:
    print("Missing:")
    print(",".join(missing))


Rows copied: 35
Missing basenames: 0


In [6]:
with pd.ExcelWriter(OUTPUT_XLSX, engine="openpyxl") as writer:
    filtered_out.to_excel(writer, index=False, sheet_name="Filtered_Records")
    missing_df.to_excel(writer, index=False, sheet_name="Missing_basenames")
    summary_df.to_excel(writer, index=False, sheet_name="Summary")

print(f"Written: {OUTPUT_XLSX.resolve()}")


Written: /home/kesju/DI/2025_ZIVEO/PROJECT_TRAIN_UNET/1_PREPARE_TRAIN_UNET_DATA/CONVERT_ZIVE_TO&FROM_NPY/filtered_records_by_basenames.xlsx


In [7]:
check = pd.read_excel(OUTPUT_XLSX, sheet_name="Summary")
check


,Parameter,Value
0,Source Excel,visi_zive_irasai_annot-Darb_26_04_16_atrankai....
1,Comma-list file,converted_basenames_comma_list.txt
2,Basename column,basename
3,Basenames requested,35
4,Rows copied,35
5,Missing basenames,0
6,Output Excel,filtered_records_by_basenames.xlsx
